# Refusals and errors

The robot says no in a few different ways, and each one has its own name. This module
makes each one happen that the mock can show, and ends with the shape of a program that
handles them.

Everything here is a `g1lab.G1LabError`, so you can catch them all or only the one you
expect:

| Exception | What it means | What to do |
|---|---|---|
| `Disconnected` | no answer from the robot computer | check the address (in the lab: `192.168.123.222`) |
| `NoSession` | the robot computer answers, but no session is running | the operator starts one; `control()` waits for it |
| `Busy` | someone else has the robot (`e.owner`, your place `e.place`) | `control()` waits your turn; this comes only when `wait` runs out |
| `NameInUse` | someone on another computer uses your name | pick another `name` |
| `Refused` | the pose would hit the robot itself (`e.reason` says what) | change that arm's angles |
| `Ended` | the session is over, or ending | stop; your program is done |
| `NoData` | a sensor sent nothing | that sensor is off, or the session does not carry it |
| `ServiceRefused` | `r.services.start()` may not start that service | read the reason; the operator starts robot services |

Run it on the mock first: parts 2 and 3 move the arms.

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import g1lab

# "mock" runs the robot on this laptop (needs `pip install mujoco`).
# In the lab, the robot computer: HOST = "192.168.123.222"
HOST = os.environ.get("G1LAB_HOST", "mock")
NAME = os.environ.get("G1LAB_NAME", "student")     # your name: it shows on the lab board

r = g1lab.mock(name=NAME) if HOST == "mock" else g1lab.connect(HOST, name=NAME)
info = r.info()
print("connected to", HOST, "| session:", info.phase, "| has the robot:", info.owner)

## 1. Disconnected and NoData

`g1lab.connect()` asks the robot computer straight away, so a wrong address raises
`Disconnected` there, after 2 seconds. `r.health()` says which sensor feeds are live when a
read raises `NoData`.

In [ ]:
try:
    g1lab.connect("10.255.255.1", name=NAME)         # nothing answers at this address
except g1lab.Disconnected as e:
    print("Disconnected:", e)

try:
    r.depth.read()
    print("depth is on")
except g1lab.NoData as e:
    print("NoData:", e)
for feed, s in r.health()["sources"].items():
    print(f"  {feed:8s} {'live' if s['fresh'] else 'stale or off'}")

## 2. Clamped: asking for more than a joint can do

This is not an error. The joint goes as far as it is allowed and the arm keeps
following you, and `c.status.clamped` names the joint. An angle list of the wrong
length, or with NaN in it, never leaves your program: it raises `ValueError`, which is
not a `G1LabError`, so a bug in your maths stays visible.

In [ ]:
home = info.home["right_arm"].copy()
lo, hi = info.limits["right_arm"].T
print("elbow limits (rad):", lo[3].round(2), "to", hi[3].round(2))

too_far = home.copy()
too_far[0] -= 0.8                            # shoulder_pitch: forward, clear of the legs
too_far[3] = -99.0                           # elbow: far past its limit

with r.control() as c:
    c.arms.move_to(right=too_far, duration=1.5)
    print("clamped:", c.status.clamped)
    print("elbow now:", r.state.joints().arm("right")[3].round(2))
    c.arms.home()

    try:
        c.arms.set(right=[0.0, 0.0, 0.0])
    except ValueError as e:
        print("ValueError:", e)

## 3. Refused: a pose that would hit the robot

The robot checks every pose against its own body. `set` and `move_to` report a refusal
differently, and on the robot the difference matters:

- **`c.arms.set(pose)`** is refused before the arm moves. It does not raise: the reason
  is on `c.status.refused`, and stays there until the robot takes another pose.
- **`c.arms.move_to(pose)`** walks the path in small steps and raises `Refused` at the
  first step that would hit something. **By then the arm has moved part of the way.**

Either way the arm holds the last safe pose, and your next `move_to` or `home()` starts
from there. Here both are asked for the same pose, which puts the two hands together.

In [ ]:
together = np.zeros(7)

with r.control() as c:
    before = r.state.joints().arm("right").copy()
    c.arms.set(right=together, left=together)
    time.sleep(0.3)
    moved = np.degrees(np.abs(r.state.joints().arm("right") - before)).max()
    print("set: refused =", c.status.refused)
    print(f"  the arm moved {moved:.1f} deg")

    c.arms.home()                            # starts from the last safe pose
    print("  after home(): refused =", c.status.refused)

    before = r.state.joints().arm("right").copy()
    try:
        c.arms.move_to(right=together, left=together, duration=2.0)
    except g1lab.Refused as e:
        moved = np.degrees(np.abs(r.state.joints().arm("right") - before)).max()
        print("move_to: Refused =", e.reason)
        print(f"  the arm moved {moved:.1f} deg before it")
    c.arms.home()
    print("  home again")

Two more answers can show on `c.status`, on the robot and in the simulator. This run
does not show them:

- **`c.status.held`**: the arm stopped part way because going further would hit something.
  It tries to go round by itself. If `held` stays, go via a pose further out to the side
  or forward.
- **`c.status.resting`**: an arm was held up for 30 s, so it went down to rest for about
  15 s, because holding it up heats the shoulder motor. Lower the arms now and then.

## 4. Ended, and the shape of a whole program

A session has an end, and the operator can end it early. Whatever call comes next raises
`Ended`, so catch it around the whole block. `wait` is how long
`control()` waits for your turn and for a session, and when it runs out you get `Busy` or
`NoSession`. On the mock, the session here lasts 6 s, so this program runs into its end.

In [ ]:
robot = g1lab.mock(name=NAME, minutes=0.1) if HOST == "mock" else r

def my_program(robot, seconds=20):
    home = robot.info().home["right_arm"].copy()
    up = home.copy()
    up[0] -= 0.5                             # shoulder_pitch: forward
    try:
        with robot.control(wait=30) as c:
            t0 = time.time()
            while time.time() - t0 < seconds:
                try:
                    c.arms.move_to(right=up, duration=1.0)
                    c.arms.move_to(right=home, duration=1.0)
                except g1lab.Refused as e:
                    print("not that pose:", e.reason)
                    break
            print("finished")
    except g1lab.Ended:
        print("the session ended: the arms went home by themselves")
    except (g1lab.Busy, g1lab.NoSession) as e:
        print("no turn this time:", e)
    except g1lab.Disconnected as e:
        print("lost the robot computer:", e)

my_program(robot)

**Try this.** Change `up` to a pose you expect to be refused, and read the reason you get.
Then replace `move_to` with `set` plus a check of `c.status.refused`, and compare how far
the arm moves before each one stops.

In [ ]:
if robot is not r:
    robot.close()
r.close()